## Question A1 (13 marks)

### 1. Draw a block diagram of the baseline model and report the number of trainable parameters in it.

In [1]:
# From src/config.py, the model is defined as:
import torch.nn as nn
import torch
class Config:
    seed: int = 42
    train_years: tuple[int, ...] = (2017, 2018, 2019, 2020)
    validation_year: int = 2021
    test_year: int = 2022

    categorical_features: tuple[str, ...] = (
        "month",
        "town",
        "flat_model_type",
        "storey_range",
    )
    embedding_dims: tuple[int, ...] = (8, 8, 8, 8)
    continuous_features: tuple[str, ...] = (
        "dist_to_nearest_stn",
        "dist_to_dhoby",
        "degree_centrality",
        "eigenvector_centrality",
        "remaining_lease_years",
        "floor_area_sqm",
    )

    hidden_width: int = 64

    learning_rate: float = 1e-2
    batch_size: int = 512
    max_epochs: int = 20
    early_stopping_patience: int = 4
    early_stopping_threshold: float = 1e-4

# and from model.py,

class PriceModel(nn.Module):
    def __init__(
        self,
        cardinalities: list[int],
        target_mean: float,
        target_std: float,
        config: Config,
    ) -> None:
        super().__init__()
        self.embeddings = nn.ModuleList(
            nn.Embedding(cardinality, width)
            for cardinality, width in zip(cardinalities, config.embedding_dims)
        )

        input_width = sum(config.embedding_dims) + len(config.continuous_features)
        self.mlp = nn.Sequential(
            nn.Linear(input_width, config.hidden_width),
            nn.LayerNorm(config.hidden_width),
            nn.ReLU(),
            nn.Linear(config.hidden_width, 1),
        )

        self.target_mean = target_mean
        self.target_std = target_std

On close inspection, we see that the model is defined with an embedding table along with a single MLP. Our first step is to determine the dimensions of these embedding tables.

When initialising the model, we take in `cardinalities` which is a list of integers. This corresponds to the number of unique values for each categorical feature, plus 1 for the unseen class. We can use pandas to open the csv and read the number of unique values:

In [2]:
import pandas as pd

ds = pd.read_csv("./hdb_price_prediction.csv")
categorical_features: tuple[str, ...] = (
        "month",
        "town",
        "flat_model_type",
        "storey_range",
)

In [3]:
ds_categorical = ds[list(categorical_features)]
ds_categorical.head()

,month,town,flat_model_type,storey_range
0,1,ANG MO KIO,"2 ROOM, Improved",10 TO 12
1,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03
2,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03
3,1,ANG MO KIO,"3 ROOM, New Generation",04 TO 06
4,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03


In [4]:
ds_categorical.nunique()

month              12
town               26
flat_model_type    44
storey_range       17
dtype: int64

Hence, the categorical cardinalities are:

`month`: 13 \
`town`: 27 \
`flat_model_type`: 44 \
`storey_range`: 17

Since the config defines the embedding dimension to be 8, this means that each `nn.Embedding` table will be of size `cardinality * 8`:

```py

self.embeddings = nn.ModuleList(
            nn.Embedding(13, 8) # month
            nn.Embedding(27, 8) # town
            nn.Embedding(44, 8) # flat_model_type
            nn.Embedding(17, 8) # story_range
        )
```

Our input width is calculated as `4 embds * 8 = 32`, plus 6 continuous features, giving us an input width of 38.

So, our network is:

```py
input_width = sum(config.embedding_dims) + len(config.continuous_features) # 4*8 + 6
self.mlp = nn.Sequential(
    nn.Linear(38, 64), # config.hidden_width is given as 64
    nn.LayerNorm(64),
    nn.ReLU(),
    nn.Linear(64, 1),
)
```
Some simple math will give us:

Embedding parameters:
$$ (13+27+44+18)\times8 = 816 \text{ embedding parameters}$$

For the first linear layer:
$$ 38 \times 64\text{ weights} + 64\text{ bias} = 2496$$

LayerNorm:
$$ 64 + 64 = 128 $$

Final output layer:
$$ 64 + 1 = 65 $$

Therefore, our total parameters are:

$$ 816 + 2496 + 128 + 65 = 3505 $$

Credit to https://github.com/WangX0111/drawio-nn-templates for the draw.io templates that are used for the report and this assignment. Final block diagram:

<img src="https://www.google.com"> </img>

### 2. Perform hyperparamter optimisation on the baseline using grid search with the Ray library

We use the Ray library to find the optimal hidden layer width and embedding dimension. The given search space is `{16, 32, 64, 128, 256}` and `{2, 4, 8, 12, 16}` respectively. This gives us 25 possible configurations to test.


In [16]:
import copy
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
import ray
from ray import tune
from ray.tune import RunConfig, Tuner

# ensure that ray can find the path, fix the import error
REPO_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

if ray.is_initialized():
    ray.shutdown()
ray.init(
    runtime_env={
        "working_dir": str(REPO_ROOT) + "/",
        "env_vars": {"PYTHONPATH": str(REPO_ROOT / "src")},
        "excludes": ["*.csv"],
    },
    ignore_reinit_error=True,
    include_dashboard=False,
)

from src.baseline import calculate_rmse
from src.config import Config
from src.data import prepare_data
from src.model import PriceModel


torch.set_num_threads(1)
torch.use_deterministic_algorithms(True, warn_only=True)

SEED = Config().seed


DATA = prepare_data("hdb_price_prediction.csv", Config())

print(f"train {len(DATA['train'])} rows")
print(f"validation {len(DATA['validation'])} rows")

2026-09-29 19:25:59,479	INFO worker.py:2024 -- Started a local Ray instance.
2026-09-29 19:25:59,482	INFO packaging.py:363 -- Ignoring upload to cluster for these files: [PosixPath('/home/aaron/sc4001/parta/.gitignore')]
2026-09-29 19:25:59,484	INFO packaging.py:363 -- Ignoring upload to cluster for these files: [PosixPath('/home/aaron/sc4001/parta/.venv/.gitignore')]
2026-09-29 19:25:59,487	INFO packaging.py:734 -- Creating a file package for local module '/home/aaron/sc4001/parta/'.
2026-09-29 19:25:59,488	INFO packaging.py:363 -- Ignoring upload to cluster for these files: [PosixPath('/home/aaron/sc4001/parta/.gitignore')]
2026-09-29 19:25:59,492	INFO packaging.py:363 -- Ignoring upload to cluster for these files: [PosixPath('/home/aaron/sc4001/parta/.venv/.gitignore')]
2026-09-29 19:25:59,494	INFO packaging.py:434 -- Pushing file package 'gcs://_ray_pkg_d0123cc6c802201f.zip' (0.44MiB) to Ray cluster...
2026-09-29 19:25:59,497	INFO packaging.py:447 -- Successfully pushed file packag

train 87370 rows
validation 29057 rows


In [17]:
import copy

from src.baseline import calculate_rmse

# a small mod to the training loop in baseline to collect history
def train_with_history(model, train_data, validation_data, config):
    train_loader = DataLoader(train_data, config.batch_size, shuffle=True)
    loss_function = nn.MSELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

    best_rmse = float("inf")
    best_weights = None
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, config.max_epochs + 1):
        model.train()
        squared_error = 0.0
        row_count = 0

        for categorical, continuous, target in train_loader:
            optimiser.zero_grad()
            standard_prediction = model(categorical, continuous)
            standard_target = (target - model.target_mean) / model.target_std
            loss = loss_function(standard_prediction, standard_target)
            loss.backward()
            optimiser.step()

            prediction = (
                standard_prediction.detach() * model.target_std + model.target_mean
            )
            squared_error += (prediction - target).square().sum().item()
            row_count += len(target)

        # train_rmse = (squared_error / row_count) ** 0.5
        validation_rmse = calculate_rmse(model, validation_data, config.batch_size)
        # print(
        #     f"epoch {epoch:02d} | train RMSE {train_rmse:,.0f} "
        #     f"| validation RMSE {validation_rmse:,.0f}"
        # )

        history.append(((squared_error / row_count) ** 0.5, validation_rmse))

        if (
            best_weights is None
            or validation_rmse < best_rmse - config.early_stopping_threshold
        ):
            best_rmse = validation_rmse
            best_weights = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == config.early_stopping_patience:
                # print(f"early stopping after epoch {epoch}")
                break

    model.load_state_dict(best_weights)
    return best_epoch, best_rmse, history


def build_model(hidden_width, embedding_dim, data):
    config = Config()
    config.hidden_width = hidden_width
    config.embedding_dims = (embedding_dim,) * len(config.categorical_features)
    model = PriceModel(
        data["cardinalities"], data["target_mean"], data["target_std"], config
    )
    return model, config


def run_trial(params, data):
    torch.manual_seed(SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True, warn_only=True)

    model, config = build_model(params["hidden_width"], params["embedding_dim"], data)
    best_epoch, best_rmse, _ = train_with_history(
        model, data["train"], data["validation"], config
    )

    tune.report(
        {
            "validation_rmse": best_rmse,
            "best_epoch": best_epoch,
            "n_parameters": sum(parameter.numel() for parameter in model.parameters()),
        }
    )


In [18]:
HIDDEN_WIDTHS = (16, 32, 64, 128, 256)
EMBEDDING_DIMS = (2, 4, 8, 12, 16)

N_CONFIGURATIONS = len(HIDDEN_WIDTHS) * len(EMBEDDING_DIMS)
print(f"total {N_CONFIGURATIONS} configurations to test")

tuner = Tuner(
    tune.with_parameters(run_trial, data=DATA),
    param_space={
        "hidden_width": tune.grid_search(list(HIDDEN_WIDTHS)),
        "embedding_dim": tune.grid_search(list(EMBEDDING_DIMS)),
    },
    tune_config=tune.TuneConfig(
        num_samples=1,
        max_concurrent_trials=6,
        metric="validation_rmse",
        mode="min",
    ),
    run_config=RunConfig(
        name="grid_search",
        storage_path=str(REPO_ROOT / "outputs" / "a1"),
    ),
)

results = tuner.fit()

2026-09-29 19:26:46,726	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/aaron/sc4001/parta/outputs/a1/grid_search' in 0.0053s.
2026-09-29 19:26:46,730	INFO tune.py:1039 -- Total run time: 45.33 seconds (45.31 seconds for the tuning loop).


In [19]:
sweep = (
    results.get_dataframe()
    .loc[
        :,
        ["config/hidden_width", "config/embedding_dim", "validation_rmse", "best_epoch", "n_parameters"],
    ]
    .sort_values("validation_rmse")
    .reset_index(drop=True)
)
sweep

,config/hidden_width,config/embedding_dim,validation_rmse,best_epoch,n_parameters
0,256,2,59792.480469,10,4813
1,128,12,60036.625000,13,8649
2,256,4,61084.445312,3,7065
3,128,4,61532.652344,4,3737
4,32,4,62762.535156,4,1241
5,32,12,62953.148438,5,3081
6,16,12,63026.390625,7,2153
7,256,16,63147.457031,5,20577
8,64,2,63364.960938,7,1357
9,64,8,63411.960938,4,3505


In [28]:
best = results.get_best_result(metric="validation_rmse", mode="min")
BEST_HIDDEN = int(best.config["hidden_width"])
BEST_EMBEDDING = int(best.config["embedding_dim"])
BEST_RMSE = float(best.metrics["validation_rmse"])

print(f"best             hidden_width={BEST_HIDDEN:>3}  embedding_dim={BEST_EMBEDDING:>2}"
      f"  validation RMSE {BEST_RMSE:>10,.0f}")

# The 5 x 5 grid as a table.
sweep.pivot(index="config/hidden_width", columns="config/embedding_dim", values="validation_rmse")

best             hidden_width=256  embedding_dim= 2  validation RMSE     59,792


config/embedding_dim,2,4,8,12,16
config/hidden_width,,,,,
16,66425.546875,68879.531250,67520.015625,63026.390625,67377.000000
32,69204.000000,62762.535156,66491.015625,62953.148438,64193.027344
64,63364.960938,64523.843750,63411.960938,64197.453125,63417.312500
128,66355.992188,61532.652344,67248.296875,60036.625000,65641.828125
256,59792.480469,61084.445312,63578.562500,65522.300781,63147.457031
